# Data Understanding and Enrichment

## Import project functions

In [35]:
from pathlib import Path
import sys
import pandas as pd


# Get the current notebook location
current_path = Path.cwd()

# The notebook is inside the notebooks folder
project_root = current_path.parent

# Add the project root to Python's search path
sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: c:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast


## Import required functions

In [4]:
from src.config import (
    RAW_DATA_DIR,
    REFERENCE_EXCEL_PATH,
    UNIFIED_EXCEL_PATH,
)

from src.data_loader import (
    convert_excel_files_to_csv,
    load_data,
)

## Convert the workbooks to CSV

In [7]:
unified_df, reference_df = convert_excel_files_to_csv()

Sheets found in unified workbook:
- ethiopia_fi_unified_data: (43, 34)
- Impact_sheet: (14, 35)

Files saved successfully:
C:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast\data\raw\ethiopia_fi_unified_data.csv
C:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast\data\raw\reference_codes.csv


## Load the CSV files

In [8]:
unified_df, reference_df = load_data()

print("Files loaded successfully.")

Files loaded successfully.


## Inspect dataset

In [9]:
print("Unified dataset shape:")
print(unified_df.shape)

print("\nReference-code dataset shape:")
print(reference_df.shape)

Unified dataset shape:
(57, 35)

Reference-code dataset shape:
(71, 4)


## empty rows

In [11]:
empty_rows = unified_df.isna().all(axis=1).sum()

print("Completely empty rows:", empty_rows)

Completely empty rows: 0


## duplicate record IDs

In [12]:
duplicate_record_ids = (
    unified_df["record_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate record IDs:",
    duplicate_record_ids
)

Duplicate record IDs: 0


# Exploratory Data Overview

In [13]:
from src.data_explorer import (
    count_records,
    get_observation_date_range,
    get_indicator_coverage,
    get_event_catalog,
    get_impact_link_details,
    get_event_impact_summary,
)

In [14]:
record_type_counts = count_records(
    unified_df,
    "record_type",
)

record_type_counts

,record_type,record_count
0,observation,30
1,impact_link,14
2,event,10
3,target,3


In [15]:
pillar_counts = count_records(
    unified_df,
    "pillar",
)

pillar_counts

,pillar,record_count
0,ACCESS,20
1,USAGE,17
2,Not assigned,10
3,GENDER,6
4,AFFORDABILITY,4


In [18]:
pillar_by_record_type = pd.crosstab(
    unified_df["record_type"],
    unified_df["pillar"].fillna(
        "Not assigned"
    ),
)

pillar_by_record_type

pillar,ACCESS,AFFORDABILITY,GENDER,Not assigned,USAGE
record_type,,,,,
event,0,0,0,10,0
impact_link,4,3,1,0,6
observation,14,1,4,0,11
target,2,0,1,0,0


In [19]:
source_type_counts = count_records(
    unified_df,
    "source_type",
)

source_type_counts

,source_type,record_count
0,operator,15
1,Not assigned,14
2,survey,10
3,regulator,7
4,research,4
5,policy,3
6,calculated,2
7,news,2


In [21]:
confidence_counts = count_records(
    unified_df,
    "confidence",
)

confidence_counts

,confidence,record_count
0,high,44
1,medium,13


In [22]:
print("RECORD TYPE COUNTS")
display(record_type_counts)

print("\nPILLAR COUNTS")
display(pillar_counts)

print("\nSOURCE TYPE COUNTS")
display(source_type_counts)

print("\nCONFIDENCE COUNTS")
display(confidence_counts)

RECORD TYPE COUNTS


,record_type,record_count
0,observation,30
1,impact_link,14
2,event,10
3,target,3



PILLAR COUNTS


,pillar,record_count
0,ACCESS,20
1,USAGE,17
2,Not assigned,10
3,GENDER,6
4,AFFORDABILITY,4



SOURCE TYPE COUNTS


,source_type,record_count
0,operator,15
1,Not assigned,14
2,survey,10
3,regulator,7
4,research,4
5,policy,3
6,calculated,2
7,news,2



CONFIDENCE COUNTS


,confidence,record_count
0,high,44
1,medium,13


## temporal range of observations

In [23]:
observation_date_range = (
    get_observation_date_range(
        unified_df
    )
)

observation_date_range

,first_observation,last_observation,observations_with_dates,observations_without_dates
0,2014-12-31,2025-12-31,30,0


## Observation count by year

In [24]:
observations = unified_df[
    unified_df["record_type"] == "observation"
].copy()

from src.data_explorer import parse_dates

observations["observation_date"] = parse_dates(
    observations["observation_date"]
)

observations["year"] = (
    observations["observation_date"]
    .dt.year
)

observations_by_year = (
    observations["year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

observations_by_year.columns = [
    "year",
    "observation_count",
]

observations_by_year

,year,observation_count
0,2014,1
1,2017,1
2,2021,5
3,2023,1
4,2024,11
5,2025,11


In [25]:
observation_range_by_pillar = (
    observations
    .groupby(
        "pillar",
        dropna=False,
    )
    .agg(
        observation_count=(
            "record_id",
            "count",
        ),
        first_date=(
            "observation_date",
            "min",
        ),
        last_date=(
            "observation_date",
            "max",
        ),
        unique_years=(
            "year",
            "nunique",
        ),
    )
    .reset_index()
)

observation_range_by_pillar

,pillar,observation_count,first_date,last_date,unique_years
0,ACCESS,14,2014-12-31,2025-12-31,6
1,AFFORDABILITY,1,2024-12-31,2024-12-31,1
2,GENDER,4,2021-12-31,2024-12-31,2
3,USAGE,11,2024-07-07,2025-07-07,2


# indicators and their coverage
## Generate the indicator coverage table

In [26]:
indicator_coverage = get_indicator_coverage(
    unified_df
)

indicator_coverage

,indicator_code,indicator,pillar,record_count,first_date,last_date,years_covered,source_count,source_types
0,ACC_4G_COV,4G Population Coverage,ACCESS,2,2023-06-30,2025-06-30,"2023, 2025",1,operator
1,ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS,3,2024-08-15,2025-05-15,"2024, 2025",3,"regulator, research"
2,ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS,2,2021-12-31,2024-11-29,"2021, 2024",2,survey
3,ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS,1,2025-12-31,2025-12-31,2025,1,research
4,ACC_OWNERSHIP,Account Ownership Rate,ACCESS,6,2014-12-31,2024-11-29,"2014, 2017, 2021, 2024",4,survey
5,AFF_DATA_INCOME,Data Affordability Index,AFFORDABILITY,1,2024-12-31,2024-12-31,2024,1,research
6,GEN_GAP_ACC,Account Ownership Gender Gap,GENDER,2,2021-12-31,2024-11-29,"2021, 2024",2,survey
7,GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER,1,2024-12-31,2024-12-31,2024,1,research
8,GEN_MM_SHARE,Female Mobile Money Account Share,GENDER,1,2024-12-31,2024-12-31,2024,1,regulator
9,USG_ACTIVE_RATE,Mobile Money Activity Rate,USAGE,1,2024-12-31,2024-12-31,2024,1,calculated


In [27]:
unique_indicator_count = (
    indicator_coverage[
        "indicator_code"
    ]
    .nunique()
)

print(
    "Unique observation indicators:",
    unique_indicator_count
)

Unique observation indicators: 19


In [28]:
indicator_coverage_sorted = (
    indicator_coverage
    .sort_values(
        "record_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

indicator_coverage_sorted[
    [
        "indicator_code",
        "indicator",
        "pillar",
        "record_count",
        "years_covered",
    ]
]

,indicator_code,indicator,pillar,record_count,years_covered
0,ACC_OWNERSHIP,Account Ownership Rate,ACCESS,6,"2014, 2017, 2021, 2024"
1,ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS,3,"2024, 2025"
2,ACC_4G_COV,4G Population Coverage,ACCESS,2,"2023, 2025"
3,ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS,2,"2021, 2024"
4,GEN_GAP_ACC,Account Ownership Gender Gap,GENDER,2,"2021, 2024"
5,USG_P2P_COUNT,P2P Transaction Count,USAGE,2,"2024, 2025"
6,ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS,1,2025
7,GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER,1,2024
8,GEN_MM_SHARE,Female Mobile Money Account Share,GENDER,1,2024
9,USG_ACTIVE_RATE,Mobile Money Activity Rate,USAGE,1,2024


# catalogued events

In [29]:
event_catalog = get_event_catalog(
    unified_df
)

event_catalog

,record_id,observation_date,category,indicator,value_text,source_type,confidence
0,EVT_0001,2021-05-17,product_launch,Telebirr Launch,Launched,operator,high
1,EVT_0009,2021-09-01,policy,NFIS-II Strategy Launch,Launched,regulator,high
2,EVT_0002,2022-08-01,market_entry,Safaricom Ethiopia Commercial Launch,Launched,news,high
3,EVT_0003,2023-08-01,product_launch,M-Pesa Ethiopia Launch,Launched,operator,high
4,EVT_0004,2024-01-01,infrastructure,Fayda Digital ID Program Rollout,Launched,regulator,high
5,EVT_0005,2024-07-29,policy,Foreign Exchange Liberalization,Implemented,regulator,high
6,EVT_0006,2024-10-01,milestone,P2P Transaction Count Surpasses ATM,Achieved,operator,high
7,EVT_0007,2025-10-27,partnership,M-Pesa EthSwitch Integration,Launched,operator,high
8,EVT_0010,2025-12-15,pricing,Safaricom Ethiopia Price Increase,Implemented,news,high
9,EVT_0008,2025-12-18,infrastructure,EthioPay Instant Payment System Launch,Launched,regulator,high


In [30]:
event_category_counts = (
    count_records(
        unified_df[
            unified_df[
                "record_type"
            ] == "event"
        ],
        "category",
    )
)

event_category_counts

,category,record_count
0,product_launch,2
1,infrastructure,2
2,policy,2
3,market_entry,1
4,milestone,1
5,partnership,1
6,pricing,1


In [31]:
event_catalog["event_year"] = (
    event_catalog["observation_date"]
    .dt.year
)

events_by_year = (
    event_catalog["event_year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

events_by_year.columns = [
    "event_year",
    "event_count",
]

events_by_year

,event_year,event_count
0,2021,2
1,2022,1
2,2023,1
3,2024,3
4,2025,3


# impact-link relationships

In [32]:
impact_link_details = (
    get_impact_link_details(
        unified_df
    )
)

impact_link_details

,record_id,parent_id,event_name,event_date,event_category,pillar,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,confidence
0,IMP_0001,EVT_0001,Telebirr Launch,2021-05-17,product_launch,ACCESS,ACC_OWNERSHIP,direct,increase,high,15.0,12.0,literature,Kenya,medium
1,IMP_0002,EVT_0001,Telebirr Launch,2021-05-17,product_launch,USAGE,USG_TELEBIRR_USERS,direct,increase,high,NaN,3.0,empirical,NaN,high
2,IMP_0003,EVT_0001,Telebirr Launch,2021-05-17,product_launch,USAGE,USG_P2P_COUNT,direct,increase,high,25.0,6.0,empirical,NaN,medium
3,IMP_0004,EVT_0002,Safaricom Ethiopia Commercial Launch,2022-08-01,market_entry,ACCESS,ACC_4G_COV,direct,increase,medium,15.0,12.0,empirical,NaN,medium
4,IMP_0005,EVT_0002,Safaricom Ethiopia Commercial Launch,2022-08-01,market_entry,AFFORDABILITY,AFF_DATA_INCOME,indirect,decrease,medium,-20.0,12.0,literature,Rwanda,medium
5,IMP_0006,EVT_0003,M-Pesa Ethiopia Launch,2023-08-01,product_launch,USAGE,USG_MPESA_USERS,direct,increase,high,NaN,3.0,empirical,NaN,high
6,IMP_0007,EVT_0003,M-Pesa Ethiopia Launch,2023-08-01,product_launch,ACCESS,ACC_MM_ACCOUNT,direct,increase,medium,5.0,6.0,theoretical,NaN,medium
7,IMP_0008,EVT_0004,Fayda Digital ID Program Rollout,2024-01-01,infrastructure,ACCESS,ACC_OWNERSHIP,enabling,increase,medium,10.0,24.0,literature,India,medium
8,IMP_0009,EVT_0004,Fayda Digital ID Program Rollout,2024-01-01,infrastructure,GENDER,GEN_GAP_ACC,indirect,decrease,medium,-5.0,24.0,literature,India,medium
9,IMP_0010,EVT_0005,Foreign Exchange Liberalization,2024-07-29,policy,AFFORDABILITY,AFF_DATA_INCOME,indirect,increase,high,30.0,3.0,empirical,NaN,high


In [33]:
event_impact_summary = (
    get_event_impact_summary(
        unified_df
    )
)

event_impact_summary

,record_id,observation_date,category,indicator,impact_link_count,affected_indicators
0,EVT_0001,2021-05-17,product_launch,Telebirr Launch,3,"ACC_OWNERSHIP, USG_P2P_COUNT, USG_TELEBIRR_USERS"
1,EVT_0009,2021-09-01,policy,NFIS-II Strategy Launch,0,None
2,EVT_0002,2022-08-01,market_entry,Safaricom Ethiopia Commercial Launch,2,"ACC_4G_COV, AFF_DATA_INCOME"
3,EVT_0003,2023-08-01,product_launch,M-Pesa Ethiopia Launch,2,"ACC_MM_ACCOUNT, USG_MPESA_USERS"
4,EVT_0004,2024-01-01,infrastructure,Fayda Digital ID Program Rollout,2,"ACC_OWNERSHIP, GEN_GAP_ACC"
5,EVT_0005,2024-07-29,policy,Foreign Exchange Liberalization,1,AFF_DATA_INCOME
6,EVT_0006,2024-10-01,milestone,P2P Transaction Count Surpasses ATM,0,None
7,EVT_0007,2025-10-27,partnership,M-Pesa EthSwitch Integration,2,"USG_MPESA_ACTIVE, USG_P2P_COUNT"
8,EVT_0010,2025-12-15,pricing,Safaricom Ethiopia Price Increase,1,AFF_DATA_INCOME
9,EVT_0008,2025-12-18,infrastructure,EthioPay Instant Payment System Launch,1,USG_P2P_COUNT


# Dataset Enrichment


In [68]:
import importlib
import src.config
%load_ext autoreload
%autoreload 2
import src.enrichment



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
from src.enrichment import (
    build_record,
    save_enrichment_records,
    validate_enrichment,
    append_enrichment,
    validate_required_fields,
    validate_record_type_rules,
    validate_impact_links,
    validate_impact_pillars,
)

from src.config import (
    ENRICHMENT_CSV_PATH,
    ENRICHED_DATA_PATH,
)

In [37]:
columns = unified_df.columns

common_fields = {
    "gender": "all",
    "location": "national",
    "collected_by": "Task1_Enrichment",
    "collection_date": "2026-07-22",
}

In [45]:
observation_records = [
    build_record(
        columns,
        "REC_0034",
        "observation",
        pillar="ACCESS",
        indicator="Account Ownership Rate",
        indicator_code="ACC_OWNERSHIP",
        indicator_direction="higher_better",
        value_numeric=14,
        value_type="percentage",
        unit="%",
        observation_date="2011-12-31",
        fiscal_year=2011,
        source_name="Global Findex 2011",
        source_type="survey",
        source_url=(
            "https://www.worldbank.org/en/"
            "publication/globalfindex/download-data"
        ),
        confidence="high",
        original_text=(
            "Account ownership rate was 14% in 2011."
        ),
        notes="Historical baseline from challenge brief.",
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0035",
        "observation",
        pillar="USAGE",
        indicator="Digital Payment Adoption Rate",
        indicator_code="USG_DIGITAL_PAYMENT_RATE",
        indicator_direction="higher_better",
        value_numeric=35,
        value_type="percentage",
        unit="%",
        observation_date="2024-11-29",
        fiscal_year=2024,
        source_name=(
            "Global Findex 2024 - Challenge Baseline"
        ),
        source_type="survey",
        source_url=(
            "https://www.worldbank.org/en/"
            "publication/globalfindex/download-data"
        ),
        confidence="medium",
        original_text=(
            "Made or received a digital payment: "
            "approximately 35%."
        ),
        notes=(
            "Provisional challenge value. Replace with the "
            "exact weighted country-level Findex estimate."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0036",
        "observation",
        pillar="USAGE",
        indicator="Total Digital Transaction Value",
        indicator_code="USG_DIGITAL_TXN_VALUE",
        indicator_direction="higher_better",
        value_numeric=9_700_000_000_000,
        value_type="currency_etb",
        unit="ETB",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url=(
            "https://nbe.gov.et/nbe_news/"
            "ethiopia-launches-phase-two-of-national-"
            "digital-payments-strategy-building-on-"
            "strong-momentum-from-phase-one/"
        ),
        confidence="high",
        original_text=(
            "ETB 9.7 trillion in digital transactions "
            "during FY2023/24."
        ),
        notes=(
            "Broad national digital transaction measure."
        ),
        **common_fields,
    ),
]

In [46]:
observation_records.extend([
    build_record(
        columns,
        "REC_0037",
        "observation",
        pillar="USAGE",
        indicator="Telebirr Registered Users",
        indicator_code="USG_TELEBIRR_USERS",
        indicator_direction="higher_better",
        value_numeric=47_500_000,
        value_type="count",
        unit="users",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name=(
            "Ethio Telecom 2023/24 "
            "Annual Business Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "ethio-telecom-2023-2024-annual-"
            "business-performance-report/5/"
        ),
        confidence="high",
        notes=(
            "Adds a prior-year value to the existing "
            "Telebirr user series."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0038",
        "observation",
        pillar="USAGE",
        indicator="Telebirr Transaction Value",
        indicator_code="USG_TELEBIRR_VALUE",
        indicator_direction="higher_better",
        value_numeric=1_810_000_000_000,
        value_type="currency_etb",
        unit="ETB",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name=(
            "Ethio Telecom 2023/24 "
            "Annual Business Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "ethio-telecom-2023-2024-annual-"
            "business-performance-report/5/"
        ),
        confidence="high",
        notes=(
            "Adds a prior-year value to the existing "
            "Telebirr transaction-value series."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0039",
        "observation",
        pillar="ACCESS",
        indicator="Telebirr Agent Count",
        indicator_code="ACC_TELEBIRR_AGENTS",
        indicator_direction="higher_better",
        value_numeric=216_000,
        value_type="count",
        unit="agents",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name=(
            "Ethio Telecom 2023/24 "
            "Annual Business Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "ethio-telecom-2023-2024-annual-"
            "business-performance-report/5/"
        ),
        confidence="high",
        notes=(
            "Agent availability is an Access enabler."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0040",
        "observation",
        pillar="USAGE",
        indicator="Telebirr Merchant Count",
        indicator_code="USG_TELEBIRR_MERCHANTS",
        indicator_direction="higher_better",
        value_numeric=196_000,
        value_type="count",
        unit="merchants",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name=(
            "Ethio Telecom 2023/24 "
            "Annual Business Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "ethio-telecom-2023-2024-annual-"
            "business-performance-report/5/"
        ),
        confidence="high",
        original_text="Over 196 thousand merchants.",
        notes=(
            "Merchant acceptance creates reasons to use "
            "digital balances instead of cashing out."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0041",
        "observation",
        pillar="ACCESS",
        indicator="Telebirr Agent Count",
        indicator_code="ACC_TELEBIRR_AGENTS",
        indicator_direction="higher_better",
        value_numeric=320_300,
        value_type="count",
        unit="agents",
        observation_date="2025-06-30",
        period_start="2024-07-01",
        period_end="2025-06-30",
        fiscal_year="FY2024/25",
        source_name=(
            "Ethio Telecom 2024/25 "
            "Annual Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "2024-25-fiscal-year-annual-performance-"
            "and-three-year-lead-growth-strategy-"
            "performance/3/"
        ),
        confidence="high",
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0042",
        "observation",
        pillar="USAGE",
        indicator="Telebirr Merchant Count",
        indicator_code="USG_TELEBIRR_MERCHANTS",
        indicator_direction="higher_better",
        value_numeric=310_100,
        value_type="count",
        unit="merchants",
        observation_date="2025-06-30",
        period_start="2024-07-01",
        period_end="2025-06-30",
        fiscal_year="FY2024/25",
        source_name=(
            "Ethio Telecom 2024/25 "
            "Annual Performance"
        ),
        source_type="operator",
        source_url=(
            "https://www.ethiotelecom.et/"
            "2024-25-fiscal-year-annual-performance-"
            "and-three-year-lead-growth-strategy-"
            "performance/3/"
        ),
        confidence="high",
        **common_fields,
    ),
])

## ATM and POS observations

In [47]:
observation_records.extend([
    build_record(
        columns,
        "REC_0043",
        "observation",
        pillar="USAGE",
        indicator="ATM Transaction Count",
        indicator_code="USG_ATM_COUNT",
        indicator_direction="neutral",
        value_numeric=94_527_740,
        value_type="count",
        unit="transactions",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name="EthSwitch FY2023/24 Results",
        source_type="operator",
        source_url=(
            "https://ethswitch.com/2024/08/08/"
            "ethswitch-holds-its-annual-staff-meeting/"
        ),
        confidence="high",
        notes=(
            "Adds the prior-year ATM value for comparison "
            "with digital P2P growth."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0044",
        "observation",
        pillar="USAGE",
        indicator="POS Transaction Count",
        indicator_code="USG_POS_COUNT",
        indicator_direction="higher_better",
        value_numeric=2_181_365,
        value_type="count",
        unit="transactions",
        observation_date="2024-06-30",
        period_start="2023-07-01",
        period_end="2024-06-30",
        fiscal_year="FY2023/24",
        source_name="EthSwitch FY2023/24 Results",
        source_type="operator",
        source_url=(
            "https://ethswitch.com/2024/08/08/"
            "ethswitch-holds-its-annual-staff-meeting/"
        ),
        confidence="high",
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0045",
        "observation",
        pillar="USAGE",
        indicator="POS Transaction Count",
        indicator_code="USG_POS_COUNT",
        indicator_direction="higher_better",
        value_numeric=2_780_000,
        value_type="count",
        unit="transactions",
        observation_date="2025-06-30",
        period_start="2024-07-01",
        period_end="2025-06-30",
        fiscal_year="FY2024/25",
        source_name="EthSwitch FY2024/25 Results",
        source_type="operator",
        source_url=(
            "https://ethswitch.com/2025/10/30/"
            "ethswitch-announced-the-completion-of-"
            "fy-2024-25-with-outstanding-results/"
        ),
        confidence="high",
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0046",
        "observation",
        pillar="USAGE",
        indicator="POS Transaction Value",
        indicator_code="USG_POS_VALUE",
        indicator_direction="higher_better",
        value_numeric=7_200_000_000,
        value_type="currency_etb",
        unit="ETB",
        observation_date="2025-06-30",
        period_start="2024-07-01",
        period_end="2025-06-30",
        fiscal_year="FY2024/25",
        source_name="EthSwitch FY2024/25 Results",
        source_type="operator",
        source_url=(
            "https://ethswitch.com/2025/10/30/"
            "ethswitch-announced-the-completion-of-"
            "fy-2024-25-with-outstanding-results/"
        ),
        confidence="high",
        **common_fields,
    ),
])

## registered-versus-active indicators

In [48]:
observation_records.extend([
    build_record(
        columns,
        "REC_0047",
        "observation",
        pillar="USAGE",
        indicator=(
            "Sector Active Mobile Money Account Rate"
        ),
        indicator_code=(
            "USG_SECTOR_ACTIVE_ACCOUNT_RATE"
        ),
        indicator_direction="higher_better",
        value_numeric=15,
        value_type="percentage",
        unit="%",
        observation_date="2025-12-09",
        fiscal_year=2025,
        source_name="NBE NDPS 2.0 Draft",
        source_type="policy",
        source_url="https://nbe.gov.et/ndps/",
        confidence="medium",
        original_text=(
            "Only 15% of accounts are active."
        ),
        notes=(
            "Sector-level strategy baseline. Confirm the "
            "denominator and reference period before modeling."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "REC_0048",
        "observation",
        pillar="USAGE",
        indicator="Active Mobile Money Agent Rate",
        indicator_code="USG_ACTIVE_AGENT_RATE",
        indicator_direction="higher_better",
        value_numeric=25,
        value_type="percentage",
        unit="%",
        observation_date="2025-12-09",
        fiscal_year=2025,
        source_name="NBE NDPS 2.0 Draft",
        source_type="policy",
        source_url="https://nbe.gov.et/ndps/",
        confidence="medium",
        original_text=(
            "Only 25% of agents are active."
        ),
        notes=(
            "Sector-level strategy baseline. Confirm the "
            "denominator and reference period before modeling."
        ),
        **common_fields,
    ),
])

In [49]:
enrichment_observations_df = pd.DataFrame(
    observation_records
)

print(
    "New observation records:",
    len(enrichment_observations_df),
)

display(
    enrichment_observations_df[
        [
            "record_id",
            "pillar",
            "indicator_code",
            "indicator",
            "value_numeric",
            "unit",
            "observation_date",
            "confidence",
        ]
    ]
)

New observation records: 15


,record_id,pillar,indicator_code,indicator,value_numeric,unit,observation_date,confidence
0,REC_0034,ACCESS,ACC_OWNERSHIP,Account Ownership Rate,14,%,2011-12-31,high
1,REC_0035,USAGE,USG_DIGITAL_PAYMENT_RATE,Digital Payment Adoption Rate,35,%,2024-11-29,medium
2,REC_0036,USAGE,USG_DIGITAL_TXN_VALUE,Total Digital Transaction Value,9700000000000,ETB,2024-06-30,high
3,REC_0037,USAGE,USG_TELEBIRR_USERS,Telebirr Registered Users,47500000,users,2024-06-30,high
4,REC_0038,USAGE,USG_TELEBIRR_VALUE,Telebirr Transaction Value,1810000000000,ETB,2024-06-30,high
5,REC_0039,ACCESS,ACC_TELEBIRR_AGENTS,Telebirr Agent Count,216000,agents,2024-06-30,high
6,REC_0040,USAGE,USG_TELEBIRR_MERCHANTS,Telebirr Merchant Count,196000,merchants,2024-06-30,high
7,REC_0041,ACCESS,ACC_TELEBIRR_AGENTS,Telebirr Agent Count,320300,agents,2025-06-30,high
8,REC_0042,USAGE,USG_TELEBIRR_MERCHANTS,Telebirr Merchant Count,310100,merchants,2025-06-30,high
9,REC_0043,USAGE,USG_ATM_COUNT,ATM Transaction Count,94527740,transactions,2024-06-30,high


## Add new events

In [50]:
event_records = [
    build_record(
        columns,
        "EVT_0011",
        "event",
        category="regulation",
        indicator=(
            "Revised Payment Instrument "
            "Issuer Directive"
        ),
        indicator_code="EVT_PII_DIRECTIVE_2023",
        value_text="Implemented",
        value_type="categorical",
        observation_date="2023-10-09",
        fiscal_year=2023,
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url=(
            "https://nbe.gov.et/nbe_news/"
            "the-national-bank-of-ethiopia-has-issued-"
            "a-revised-directive-for-mobile-money-"
            "providers-to-promote-safety-competition-"
            "and-innovation/"
        ),
        confidence="high",
        notes=(
            "Expanded the enabling framework for mobile "
            "money providers and raised transaction limits."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "EVT_0012",
        "event",
        category="policy",
        indicator=(
            "National Financial Education Module Launch"
        ),
        indicator_code="EVT_FIN_EDUCATION_MODULE",
        value_text="Launched",
        value_type="categorical",
        observation_date="2024-02-29",
        fiscal_year=2024,
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url=(
            "https://nbe.gov.et/nbe_news/"
            "nbe-launches-national-financial-education-"
            "module-to-empower-youth-msms/"
        ),
        confidence="high",
        notes=(
            "Focused on youth, MSMEs, digital banking, "
            "savings, lending and financial capability."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "EVT_0013",
        "event",
        category="regulation",
        indicator=(
            "National Interoperable QR "
            "Payment Standard Launch"
        ),
        indicator_code="EVT_INTEROPERABLE_QR",
        value_text="Launched",
        value_type="categorical",
        observation_date="2024-04-05",
        fiscal_year=2024,
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url="https://nbe.gov.et/edpc/",
        confidence="high",
        notes=(
            "Introduced a unified QR standard for "
            "interoperable merchant payments."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "EVT_0014",
        "event",
        category="policy",
        indicator=(
            "National Digital Payments Strategy "
            "Phase Two Launch"
        ),
        indicator_code="EVT_NDPS_PHASE2",
        value_text="Launched",
        value_type="categorical",
        observation_date="2025-03-28",
        period_start="2025-03-28",
        period_end="2029-12-31",
        fiscal_year=2025,
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url=(
            "https://nbe.gov.et/nbe_news/"
            "ethiopia-launches-phase-two-of-national-"
            "digital-payments-strategy-building-on-"
            "strong-momentum-from-phase-one/"
        ),
        confidence="high",
        notes=(
            "Focuses on deeper usage, interoperability, "
            "digital ID and merchant acceptance."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "EVT_0015",
        "event",
        category="infrastructure",
        indicator=(
            "ISO 20022 EATS Payment System Upgrade"
        ),
        indicator_code="EVT_EATS_ISO20022",
        value_text="Implemented",
        value_type="categorical",
        observation_date="2025-03-29",
        fiscal_year=2025,
        source_name="National Bank of Ethiopia",
        source_type="regulator",
        source_url=(
            "https://nbe.gov.et/nbe_news/"
            "the-national-bank-of-ethiopia-launches-"
            "iso-20022-upgraded-ethiopian-automated-"
            "transfer-system-eats/"
        ),
        confidence="high",
        notes=(
            "Improved the security, reliability and "
            "efficiency of interbank settlement."
        ),
        **common_fields,
    ),
]

In [51]:
enrichment_events_df = pd.DataFrame(
    event_records
)

display(
    enrichment_events_df[
        [
            "record_id",
            "observation_date",
            "category",
            "indicator",
            "source_name",
            "confidence",
        ]
    ]
)

,record_id,observation_date,category,indicator,source_name,confidence
0,EVT_0011,2023-10-09,regulation,Revised Payment Instrument Issuer Directive,National Bank of Ethiopia,high
1,EVT_0012,2024-02-29,policy,National Financial Education Module Launch,National Bank of Ethiopia,high
2,EVT_0013,2024-04-05,regulation,National Interoperable QR Payment Standard Launch,National Bank of Ethiopia,high
3,EVT_0014,2025-03-28,policy,National Digital Payments Strategy Phase Two L...,National Bank of Ethiopia,high
4,EVT_0015,2025-03-29,infrastructure,ISO 20022 EATS Payment System Upgrade,National Bank of Ethiopia,high


## Add impact-link

In [52]:
impact_link_records = [
    build_record(
        columns,
        "IMP_0015",
        "impact_link",
        parent_id="EVT_0011",
        pillar="ACCESS",
        indicator=(
            "Revised mobile money directive effect "
            "on mobile money ownership"
        ),
        observation_date="2023-10-09",
        related_indicator="ACC_MM_ACCOUNT",
        relationship_type="direct",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=6,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "More enabling regulation and higher limits "
            "may support wallet acquisition."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0016",
        "impact_link",
        parent_id="EVT_0011",
        pillar="USAGE",
        indicator=(
            "Revised mobile money directive effect "
            "on active accounts"
        ),
        observation_date="2023-10-09",
        related_indicator=(
            "USG_SECTOR_ACTIVE_ACCOUNT_RATE"
        ),
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=12,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "Additional services and higher limits may "
            "support continued wallet usage."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0017",
        "impact_link",
        parent_id="EVT_0012",
        pillar="USAGE",
        indicator=(
            "Financial education effect on "
            "digital payment adoption"
        ),
        observation_date="2024-02-29",
        related_indicator="USG_DIGITAL_PAYMENT_RATE",
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="low",
        lag_months=12,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "Improved financial capability may increase "
            "safe and confident digital-service usage."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0018",
        "impact_link",
        parent_id="EVT_0013",
        pillar="USAGE",
        indicator=(
            "Interoperable QR standard effect "
            "on POS transactions"
        ),
        observation_date="2024-04-05",
        related_indicator="USG_POS_COUNT",
        relationship_type="direct",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=6,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "One interoperable QR standard reduces "
            "merchant and customer payment friction."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0019",
        "impact_link",
        parent_id="EVT_0013",
        pillar="USAGE",
        indicator=(
            "Interoperable QR standard effect on "
            "digital payment adoption"
        ),
        observation_date="2024-04-05",
        related_indicator="USG_DIGITAL_PAYMENT_RATE",
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=12,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "Greater merchant acceptance may turn "
            "account access into regular payment usage."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0020",
        "impact_link",
        parent_id="EVT_0014",
        pillar="ACCESS",
        indicator=(
            "NDPS Phase Two effect on account ownership"
        ),
        observation_date="2025-03-28",
        related_indicator="ACC_OWNERSHIP",
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=12,
        evidence_basis="expert",
        confidence="medium",
        notes=(
            "The strategy prioritizes rural populations, "
            "women, digital ID and inclusive services."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0021",
        "impact_link",
        parent_id="EVT_0014",
        pillar="USAGE",
        indicator=(
            "NDPS Phase Two effect on "
            "digital payment adoption"
        ),
        observation_date="2025-03-28",
        related_indicator="USG_DIGITAL_PAYMENT_RATE",
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="medium",
        lag_months=12,
        evidence_basis="expert",
        confidence="medium",
        notes=(
            "The strategy explicitly prioritizes "
            "usage, interoperability and merchant adoption."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0022",
        "impact_link",
        parent_id="EVT_0015",
        pillar="USAGE",
        indicator=(
            "ISO 20022 EATS upgrade effect on "
            "digital transaction value"
        ),
        observation_date="2025-03-29",
        related_indicator="USG_DIGITAL_TXN_VALUE",
        relationship_type="enabling",
        impact_direction="increase",
        impact_magnitude="low",
        lag_months=6,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "More efficient and reliable settlement may "
            "support growth in digital transaction value."
        ),
        **common_fields,
    ),

    build_record(
        columns,
        "IMP_0023",
        "impact_link",
        parent_id="EVT_0015",
        pillar="USAGE",
        indicator=(
            "ISO 20022 EATS upgrade effect "
            "on interoperable P2P activity"
        ),
        observation_date="2025-03-29",
        related_indicator="USG_P2P_COUNT",
        relationship_type="indirect",
        impact_direction="increase",
        impact_magnitude="low",
        lag_months=12,
        evidence_basis="theoretical",
        confidence="medium",
        notes=(
            "The infrastructure upgrade may improve the "
            "broader settlement environment supporting P2P."
        ),
        **common_fields,
    ),
]

In [54]:
enrichment_impact_links_df = pd.DataFrame(
    impact_link_records
)

display(
    enrichment_impact_links_df[
        [
            "record_id",
            "parent_id",
            "pillar",
            "related_indicator",
            "relationship_type",
            "impact_direction",
            "impact_magnitude",
            "lag_months",
            "confidence",
        ]
    ]
)

,record_id,parent_id,pillar,related_indicator,relationship_type,impact_direction,impact_magnitude,lag_months,confidence
0,IMP_0015,EVT_0011,ACCESS,ACC_MM_ACCOUNT,direct,increase,medium,6,medium
1,IMP_0016,EVT_0011,USAGE,USG_SECTOR_ACTIVE_ACCOUNT_RATE,enabling,increase,medium,12,medium
2,IMP_0017,EVT_0012,USAGE,USG_DIGITAL_PAYMENT_RATE,enabling,increase,low,12,medium
3,IMP_0018,EVT_0013,USAGE,USG_POS_COUNT,direct,increase,medium,6,medium
4,IMP_0019,EVT_0013,USAGE,USG_DIGITAL_PAYMENT_RATE,enabling,increase,medium,12,medium
5,IMP_0020,EVT_0014,ACCESS,ACC_OWNERSHIP,enabling,increase,medium,12,medium
6,IMP_0021,EVT_0014,USAGE,USG_DIGITAL_PAYMENT_RATE,enabling,increase,medium,12,medium
7,IMP_0022,EVT_0015,USAGE,USG_DIGITAL_TXN_VALUE,enabling,increase,low,6,medium
8,IMP_0023,EVT_0015,USAGE,USG_P2P_COUNT,indirect,increase,low,12,medium


In [76]:
COLLECTED_BY = "Bemnet Negash"
COLLECTION_DATE = "2026-07-22"

In [77]:
WORLD_BANK_FINDEX_URL = (
    "https://www.worldbank.org/en/"
    "publication/globalfindex/download-data"
)

NBE_PHASE_TWO_URL = (
    "https://nbe.gov.et/nbe_news/"
    "ethiopia-launches-phase-two-of-national-"
    "digital-payments-strategy-building-on-"
    "strong-momentum-from-phase-one/"
)

ETHIO_TELECOM_2024_URL = (
    "https://www.ethiotelecom.et/"
    "ethio-telecom-2023-2024-annual-"
    "business-performance-report/5/"
)

ETHIO_TELECOM_2025_URL = (
    "https://www.ethiotelecom.et/"
    "2024-25-fiscal-year-annual-performance-"
    "and-three-year-lead-growth-strategy-"
    "performance/3/"
)

ETHSWITCH_2024_URL = (
    "https://ethswitch.com/2024/08/08/"
    "ethswitch-holds-its-annual-staff-meeting/"
)

ETHSWITCH_2025_URL = (
    "https://ethswitch.com/2025/10/30/"
    "ethswitch-announced-the-completion-of-"
    "fy-2024-25-with-outstanding-results/"
)

NBE_NDPS_URL = (
    "https://nbe.gov.et/ndps/"
)

NBE_MOBILE_MONEY_DIRECTIVE_URL = (
    "https://nbe.gov.et/nbe_news/"
    "the-national-bank-of-ethiopia-has-issued-"
    "a-revised-directive-for-mobile-money-"
    "providers-to-promote-safety-competition-"
    "and-innovation/"
)

NBE_FINANCIAL_EDUCATION_URL = (
    "https://nbe.gov.et/nbe_news/"
    "nbe-launches-national-financial-education-"
    "module-to-empower-youth-msms/"
)

NBE_QR_URL = (
    "https://nbe.gov.et/eipqrc/"
)

NBE_ISO_20022_URL = (
    "https://nbe.gov.et/nbe_news/"
    "the-national-bank-of-ethiopia-launches-"
    "iso-20022-upgraded-ethiopian-automated-"
    "transfer-system-eats/"
)

In [78]:
observation_documentation = {
    "REC_0034": {
        "source_url": WORLD_BANK_FINDEX_URL,
        "original_text": "14%",
        "confidence": "high",
        "notes": (
            "Adds the missing 2011 account ownership baseline "
            "and extends the Access time series."
        ),
    },

    "REC_0035": {
        "source_url": WORLD_BANK_FINDEX_URL,
        "original_text": "approximately 35%",
        "confidence": "medium",
        "notes": (
            "Introduces the primary Usage outcome required by "
            "the challenge. The exact weighted Findex estimate "
            "should be confirmed before final modelling."
        ),
    },

    "REC_0036": {
        "source_url": NBE_PHASE_TWO_URL,
        "original_text": "9.7 trillion Birr in digital transactions",
        "confidence": "high",
        "notes": (
            "Provides a national measure of the scale of digital "
            "payment activity during FY2023/24."
        ),
    },

    "REC_0037": {
        "source_url": ETHIO_TELECOM_2024_URL,
        "original_text": "47.5 million",
        "confidence": "high",
        "notes": (
            "Adds a prior-year Telebirr subscriber observation "
            "for measuring registration growth."
        ),
    },

    "REC_0038": {
        "source_url": ETHIO_TELECOM_2024_URL,
        "original_text": "1.81 trillion-birr",
        "confidence": "high",
        "notes": (
            "Adds a prior-year Telebirr transaction-value "
            "observation for evaluating usage growth."
        ),
    },

    "REC_0039": {
        "source_url": ETHIO_TELECOM_2024_URL,
        "original_text": "216 thousand",
        "confidence": "high",
        "notes": (
            "Agent availability is an Access enabler, especially "
            "where formal bank branches are limited."
        ),
    },

    "REC_0040": {
        "source_url": ETHIO_TELECOM_2024_URL,
        "original_text": "over 196 thousand merchants",
        "confidence": "high",
        "notes": (
            "Merchant acceptance creates practical reasons for "
            "customers to retain and use digital balances."
        ),
    },

    "REC_0041": {
        "source_url": ETHIO_TELECOM_2025_URL,
        "original_text": "320.3 thousand",
        "confidence": "high",
        "notes": (
            "Extends the Telebirr agent series to FY2024/25 and "
            "supports analysis of access-point expansion."
        ),
    },

    "REC_0042": {
        "source_url": ETHIO_TELECOM_2025_URL,
        "original_text": "310.1 thousand",
        "confidence": "high",
        "notes": (
            "Extends the merchant network series and helps assess "
            "growth in digital-payment acceptance."
        ),
    },

    "REC_0043": {
        "source_url": ETHSWITCH_2024_URL,
        "original_text": "94,527,740 ATM transactions",
        "confidence": "high",
        "notes": (
            "Provides the FY2023/24 ATM baseline needed to compare "
            "cash-channel activity with digital P2P growth."
        ),
    },

    "REC_0044": {
        "source_url": ETHSWITCH_2024_URL,
        "original_text": "2,181,365 transactions",
        "confidence": "high",
        "notes": (
            "Provides a merchant-payment usage indicator for "
            "FY2023/24."
        ),
    },

    "REC_0045": {
        "source_url": ETHSWITCH_2025_URL,
        "original_text": "2.78 million interoperable POS transactions",
        "confidence": "high",
        "notes": (
            "Extends the POS transaction-count series to FY2024/25."
        ),
    },

    "REC_0046": {
        "source_url": ETHSWITCH_2025_URL,
        "original_text": "ETB 7.2 billion",
        "confidence": "high",
        "notes": (
            "Adds the value of POS transactions and complements "
            "the POS transaction-count indicator."
        ),
    },

    "REC_0047": {
        "source_url": NBE_NDPS_URL,
        "original_text": "Only 15% of accounts",
        "confidence": "medium",
        "notes": (
            "Captures the registered-versus-active account gap. "
            "The denominator and exact reporting period should be "
            "verified before using it as a model target."
        ),
    },

    "REC_0048": {
        "source_url": NBE_NDPS_URL,
        "original_text": "25% of agents are active",
        "confidence": "medium",
        "notes": (
            "Shows that the number of registered agents may "
            "overstate the number providing active services."
        ),
    },
}

In [79]:
event_documentation = {
    "EVT_0011": {
        "source_url": NBE_MOBILE_MONEY_DIRECTIVE_URL,
        "original_text": "a more enabling regulatory framework",
        "confidence": "high",
        "notes": (
            "The directive may affect mobile money Access and "
            "Usage by changing limits, services and provider rules."
        ),
    },

    "EVT_0012": {
        "source_url": NBE_FINANCIAL_EDUCATION_URL,
        "original_text": (
            "Standardized National Financial Education Module"
        ),
        "confidence": "high",
        "notes": (
            "Financial capability may influence whether people "
            "understand, trust and regularly use digital services."
        ),
    },

    "EVT_0013": {
        "source_url": NBE_QR_URL,
        "original_text": (
            "national standard for interoperable QR code payments"
        ),
        "confidence": "high",
        "notes": (
            "Interoperable QR payments can expand merchant "
            "acceptance and reduce provider-specific payment friction."
        ),
    },

    "EVT_0014": {
        "source_url": NBE_PHASE_TWO_URL,
        "original_text": "launch of Phase Two",
        "confidence": "high",
        "notes": (
            "The strategy is relevant because it prioritizes usage, "
            "interoperability, digital ID and merchant acceptance."
        ),
    },

    "EVT_0015": {
        "source_url": NBE_ISO_20022_URL,
        "original_text": "effective since March 29, 2025",
        "confidence": "high",
        "notes": (
            "The EATS upgrade may improve the reliability and "
            "efficiency of Ethiopia's payment infrastructure."
        ),
    },
}

In [80]:
impact_link_documentation = {
    "IMP_0015": {
        "source_url": NBE_MOBILE_MONEY_DIRECTIVE_URL,
        "original_text": "support financial inclusion",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: more enabling mobile money rules "
            "may increase mobile money account ownership. The source "
            "does not provide a measured effect size."
        ),
    },

    "IMP_0016": {
        "source_url": NBE_MOBILE_MONEY_DIRECTIVE_URL,
        "original_text": "expanding mobile money services",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: expanded services and transaction "
            "limits may encourage continued account activity."
        ),
    },

    "IMP_0017": {
        "source_url": NBE_FINANCIAL_EDUCATION_URL,
        "original_text": "essential financial knowledge and skills",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: improved financial capability may "
            "support safer and more confident digital-payment usage."
        ),
    },

    "IMP_0018": {
        "source_url": NBE_QR_URL,
        "original_text": "Encourage Digital Payments",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: an interoperable QR standard may "
            "increase merchant-payment and POS transaction activity."
        ),
    },

    "IMP_0019": {
        "source_url": NBE_QR_URL,
        "original_text": "Enhance Financial Access",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: wider merchant acceptance may "
            "convert account ownership into regular payment usage."
        ),
    },

    "IMP_0020": {
        "source_url": NBE_PHASE_TWO_URL,
        "original_text": "expanding digital ID integration",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: stronger digital ID integration and "
            "inclusive services may reduce barriers to account access."
        ),
    },

    "IMP_0021": {
        "source_url": NBE_PHASE_TWO_URL,
        "original_text": "deepening usage of digital payments",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: NDPS Phase Two may support higher "
            "digital-payment adoption over the strategy period."
        ),
    },

    "IMP_0022": {
        "source_url": NBE_ISO_20022_URL,
        "original_text": (
            "enhancing the efficiency, security, and reliability"
        ),
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: more reliable settlement "
            "infrastructure may support digital transaction growth."
        ),
    },

    "IMP_0023": {
        "source_url": NBE_ISO_20022_URL,
        "original_text": "optimize operational efficiency",
        "confidence": "medium",
        "notes": (
            "Modeled hypothesis: improved settlement efficiency may "
            "indirectly support interoperable P2P transactions."
        ),
    },
}

In [81]:
record_documentation = {
    **observation_documentation,
    **event_documentation,
    **impact_link_documentation,
}

print(
    "Documented record IDs:",
    len(record_documentation),
)

Documented record IDs: 29


## Combine and save the enrichment records

In [82]:
all_enrichment_records = (
    observation_records
    + event_records
    + impact_link_records
)


for record in all_enrichment_records:

    record_id = record["record_id"]

    if record_id not in record_documentation:
        print(
            "Documentation missing for:",
            record_id,
        )
        continue

    # Apply record-specific documentation
    record.update(
        record_documentation[record_id]
    )

    # Apply common documentation
    record["collected_by"] = COLLECTED_BY
    record["collection_date"] = COLLECTION_DATE

In [83]:
enrichment_df = pd.DataFrame(
    all_enrichment_records
)

documentation_fields = [
    "source_url",
    "original_text",
    "confidence",
    "collected_by",
    "collection_date",
    "notes",
]

documentation_summary = []

for field in documentation_fields:

    missing_count = (
        enrichment_df[field]
        .isna()
        .sum()
    )

    blank_count = (
        enrichment_df[field]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    documentation_summary.append({
        "field": field,
        "missing_or_blank": max(
            missing_count,
            blank_count,
        ),
    })


documentation_summary_df = pd.DataFrame(
    documentation_summary
)

documentation_summary_df

,field,missing_or_blank
0,source_url,0
1,original_text,0
2,confidence,0
3,collected_by,0
4,collection_date,0
5,notes,0


In [84]:
invalid_source_urls = enrichment_df[
    ~enrichment_df["source_url"]
    .fillna("")
    .str.startswith(
        (
            "http://",
            "https://",
        )
    )
][
    [
        "record_id",
        "record_type",
        "source_url",
    ]
]

print(
    "Records with invalid source URLs:",
    len(invalid_source_urls),
)

invalid_source_urls

Records with invalid source URLs: 0


,record_id,record_type,source_url


In [86]:
documentation_table = enrichment_df[
    [
        "record_id",
        "record_type",
        "indicator",
        "source_url",
        "original_text",
        "confidence",
        "collected_by",
        "collection_date",
        "notes",
    ]
].copy()

documentation_table

,record_id,record_type,indicator,source_url,original_text,confidence,collected_by,collection_date,notes
0,REC_0034,observation,Account Ownership Rate,https://www.worldbank.org/en/publication/globa...,14%,high,Bemnet Negash,2026-07-22,Adds the missing 2011 account ownership baseli...
1,REC_0035,observation,Digital Payment Adoption Rate,https://www.worldbank.org/en/publication/globa...,approximately 35%,medium,Bemnet Negash,2026-07-22,Introduces the primary Usage outcome required ...
2,REC_0036,observation,Total Digital Transaction Value,https://nbe.gov.et/nbe_news/ethiopia-launches-...,9.7 trillion Birr in digital transactions,high,Bemnet Negash,2026-07-22,Provides a national measure of the scale of di...
3,REC_0037,observation,Telebirr Registered Users,https://www.ethiotelecom.et/ethio-telecom-2023...,47.5 million,high,Bemnet Negash,2026-07-22,Adds a prior-year Telebirr subscriber observat...
4,REC_0038,observation,Telebirr Transaction Value,https://www.ethiotelecom.et/ethio-telecom-2023...,1.81 trillion-birr,high,Bemnet Negash,2026-07-22,Adds a prior-year Telebirr transaction-value o...
5,REC_0039,observation,Telebirr Agent Count,https://www.ethiotelecom.et/ethio-telecom-2023...,216 thousand,high,Bemnet Negash,2026-07-22,"Agent availability is an Access enabler, espec..."
6,REC_0040,observation,Telebirr Merchant Count,https://www.ethiotelecom.et/ethio-telecom-2023...,over 196 thousand merchants,high,Bemnet Negash,2026-07-22,Merchant acceptance creates practical reasons ...
7,REC_0041,observation,Telebirr Agent Count,https://www.ethiotelecom.et/2024-25-fiscal-yea...,320.3 thousand,high,Bemnet Negash,2026-07-22,Extends the Telebirr agent series to FY2024/25...
8,REC_0042,observation,Telebirr Merchant Count,https://www.ethiotelecom.et/2024-25-fiscal-yea...,310.1 thousand,high,Bemnet Negash,2026-07-22,Extends the merchant network series and helps ...
9,REC_0043,observation,ATM Transaction Count,https://ethswitch.com/2024/08/08/ethswitch-hol...,"94,527,740 ATM transactions",high,Bemnet Negash,2026-07-22,Provides the FY2023/24 ATM baseline needed to ...


In [87]:
enrichment_df = save_enrichment_records(
    all_enrichment_records
)

print(
    "Enrichment records saved to:",
    ENRICHMENT_CSV_PATH,
)

print(
    "Enrichment dataset shape:",
    enrichment_df.shape,
)

Enrichment records saved to: C:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast\data\raw\enrichment_records.csv
Enrichment dataset shape: (29, 35)


## Validate the enrichment records

In [58]:
validation_results = validate_enrichment(
    unified_df,
    enrichment_df,
)

validation_results

,check,issues_found,status
0,Duplicate IDs inside enrichment data,0,Passed
1,Enrichment IDs already in starter data,0,Passed
2,Invalid record types,0,Passed
3,Events incorrectly assigned to pillars,0,Passed
4,Impact links without matching events,0,Passed
5,Impact links with unknown indicators,0,Passed


## processed dataset

In [59]:
total_validation_issues = (
    validation_results["issues_found"]
    .sum()
)

if total_validation_issues == 0:
    enriched_df = append_enrichment(
        unified_df,
        enrichment_df,
    )

    print("Enrichment completed successfully.")
    print("Processed file:", ENRICHED_DATA_PATH)

else:
    print(
        "Enrichment was not applied because "
        "validation issues were found."
    )

Enrichment completed successfully.
Processed file: C:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast\data\processed\ethiopia_fi_enriched.csv


In [89]:
valid_confidence_values = {
    "high",
    "medium",
    "low",
}

invalid_confidence = enrichment_df[
    ~enrichment_df["confidence"]
    .isin(valid_confidence_values)
][
    [
        "record_id",
        "record_type",
        "confidence",
    ]
]

print(
    "Records with invalid confidence:",
    len(invalid_confidence),
)

invalid_confidence

Records with invalid confidence: 0


,record_id,record_type,confidence


In [90]:
documentation_issue_count = (
    documentation_summary_df[
        "missing_or_blank"
    ].sum()
)

invalid_url_count = len(
    invalid_source_urls
)

invalid_confidence_count = len(
    invalid_confidence
)

In [61]:
before_counts = (
    unified_df["record_type"]
    .value_counts()
    .rename("before_enrichment")
)

after_counts = (
    enriched_df["record_type"]
    .value_counts()
    .rename("after_enrichment")
)

record_type_comparison = (
    pd.concat(
        [
            before_counts,
            after_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

record_type_comparison.columns = [
    "record_type",
    "before_enrichment",
    "after_enrichment",
]

record_type_comparison[
    "records_added"
] = (
    record_type_comparison[
        "after_enrichment"
    ]
    - record_type_comparison[
        "before_enrichment"
    ]
)

record_type_comparison

,record_type,before_enrichment,after_enrichment,records_added
0,observation,30,45,15
1,impact_link,14,23,9
2,event,10,15,5
3,target,3,3,0


## Dataset shape comparison

In [62]:
print(
    "Starter dataset shape:",
    unified_df.shape,
)

print(
    "Enrichment dataset shape:",
    enrichment_df.shape,
)

print(
    "Processed enriched shape:",
    enriched_df.shape,
)

Starter dataset shape: (57, 35)
Enrichment dataset shape: (29, 35)
Processed enriched shape: (86, 35)


## improved indicator coverage

In [63]:
enriched_indicator_coverage = (
    get_indicator_coverage(
        enriched_df
    )
)

affected_indicator_codes = (
    enrichment_observations_df[
        "indicator_code"
    ]
    .dropna()
    .unique()
)

enriched_indicator_coverage = (
    enriched_indicator_coverage[
        enriched_indicator_coverage[
            "indicator_code"
        ].isin(affected_indicator_codes)
    ]
    .reset_index(drop=True)
)

enriched_indicator_coverage[
    [
        "indicator_code",
        "indicator",
        "pillar",
        "record_count",
        "first_date",
        "last_date",
        "years_covered",
    ]
]

,indicator_code,indicator,pillar,record_count,first_date,last_date,years_covered
0,ACC_OWNERSHIP,Account Ownership Rate,ACCESS,7,2011-12-31,2024-11-29,"2011, 2014, 2017, 2021, 2024"
1,ACC_TELEBIRR_AGENTS,Telebirr Agent Count,ACCESS,2,2024-06-30,2025-06-30,"2024, 2025"
2,USG_ACTIVE_AGENT_RATE,Active Mobile Money Agent Rate,USAGE,1,2025-12-09,2025-12-09,2025
3,USG_ATM_COUNT,ATM Transaction Count,USAGE,2,2024-06-30,2025-07-07,"2024, 2025"
4,USG_DIGITAL_PAYMENT_RATE,Digital Payment Adoption Rate,USAGE,1,2024-11-29,2024-11-29,2024
5,USG_DIGITAL_TXN_VALUE,Total Digital Transaction Value,USAGE,1,2024-06-30,2024-06-30,2024
6,USG_POS_COUNT,POS Transaction Count,USAGE,2,2024-06-30,2025-06-30,"2024, 2025"
7,USG_POS_VALUE,POS Transaction Value,USAGE,1,2025-06-30,2025-06-30,2025
8,USG_SECTOR_ACTIVE_ACCOUNT_RATE,Sector Active Mobile Money Account Rate,USAGE,1,2025-12-09,2025-12-09,2025
9,USG_TELEBIRR_MERCHANTS,Telebirr Merchant Count,USAGE,2,2024-06-30,2025-06-30,"2024, 2025"


In [64]:
starter_indicator_codes = set(
    unified_df.loc[
        unified_df["record_type"]
        == "observation",
        "indicator_code",
    ].dropna()
)

enriched_indicator_codes = set(
    enriched_df.loc[
        enriched_df["record_type"]
        == "observation",
        "indicator_code",
    ].dropna()
)

new_indicator_codes = sorted(
    enriched_indicator_codes
    - starter_indicator_codes
)

print(
    "New unique observation indicators:",
    len(new_indicator_codes),
)

for indicator_code in new_indicator_codes:
    print("-", indicator_code)

New unique observation indicators: 8
- ACC_TELEBIRR_AGENTS
- USG_ACTIVE_AGENT_RATE
- USG_DIGITAL_PAYMENT_RATE
- USG_DIGITAL_TXN_VALUE
- USG_POS_COUNT
- USG_POS_VALUE
- USG_SECTOR_ACTIVE_ACCOUNT_RATE
- USG_TELEBIRR_MERCHANTS


In [66]:
print("=" * 60)
print("CHECKPOINT 3 SUMMARY")
print("=" * 60)

print(
    "\nStarter records:",
    len(unified_df),
)

print(
    "New observations:",
    len(observation_records),
)

print(
    "New events:",
    len(event_records),
)

print(
    "New impact links:",
    len(impact_link_records),
)

print(
    "Total enrichment records:",
    len(enrichment_df),
)

print(
    "Final enriched records:",
    len(enriched_df),
)

print(
    "New unique observation indicators:",
    len(new_indicator_codes),
)

print(
    "Validation issues:",
    total_validation_issues,
)

print(
    "\nProcessed dataset:"
)

print(ENRICHED_DATA_PATH)

CHECKPOINT 3 SUMMARY

Starter records: 57
New observations: 15
New events: 5
New impact links: 9
Total enrichment records: 29
Final enriched records: 86
New unique observation indicators: 8
Validation issues: 0

Processed dataset:
C:\Users\bemnet\Desktop\10academy\newWeek11\new-ethiopia-fi-forcast\data\processed\ethiopia_fi_enriched.csv


## Validation

In [70]:
required_field_results = (
    validate_required_fields(
        enrichment_df
    )
)

required_field_results

,record_type,field,missing_count,status
0,observation,record_id,0,Passed
1,observation,record_type,0,Passed
2,observation,pillar,0,Passed
3,observation,indicator,0,Passed
4,observation,indicator_code,0,Passed
5,observation,value_numeric,0,Passed
6,observation,observation_date,0,Passed
7,observation,source_name,0,Passed
8,observation,source_url,0,Passed
9,observation,confidence,0,Passed


In [71]:
record_type_rule_results = (
    validate_record_type_rules(
        enrichment_df
    )
)

record_type_rule_results

,check,issues_found,status
0,Observations with category assigned,0,Passed
1,Events with pillar assigned,0,Passed
2,Impact links with category assigned,0,Passed


In [72]:
impact_reference_results = (
    validate_impact_links(
        unified_df,
        enrichment_df,
    )
)

impact_reference_results

,check,issues_found,status
0,Impact links with invalid parent_id,0,Passed
1,Impact links with invalid related_indicator,0,Passed


In [73]:
impact_pillar_results = (
    validate_impact_pillars(
        unified_df,
        enrichment_df,
    )
)

impact_pillar_results

,record_id,parent_id,related_indicator,pillar,expected_pillar,pillar_matches
20,IMP_0015,EVT_0011,ACC_MM_ACCOUNT,ACCESS,ACCESS,True
21,IMP_0016,EVT_0011,USG_SECTOR_ACTIVE_ACCOUNT_RATE,USAGE,USAGE,True
22,IMP_0017,EVT_0012,USG_DIGITAL_PAYMENT_RATE,USAGE,USAGE,True
23,IMP_0018,EVT_0013,USG_POS_COUNT,USAGE,USAGE,True
24,IMP_0019,EVT_0013,USG_DIGITAL_PAYMENT_RATE,USAGE,USAGE,True
25,IMP_0020,EVT_0014,ACC_OWNERSHIP,ACCESS,ACCESS,True
26,IMP_0021,EVT_0014,USG_DIGITAL_PAYMENT_RATE,USAGE,USAGE,True
27,IMP_0022,EVT_0015,USG_DIGITAL_TXN_VALUE,USAGE,USAGE,True
28,IMP_0023,EVT_0015,USG_P2P_COUNT,USAGE,USAGE,True


In [91]:
required_field_issue_count = (
    required_field_results[
        "missing_count"
    ].sum()
)

record_type_issue_count = (
    record_type_rule_results[
        "issues_found"
    ].sum()
)

impact_reference_issue_count = (
    impact_reference_results[
        "issues_found"
    ].sum()
)

impact_pillar_issue_count = (
    (~impact_pillar_results[
        "pillar_matches"
    ]).sum()
)

total_validation_issues = (
    required_field_issue_count
    + record_type_issue_count
    + impact_reference_issue_count
    + impact_pillar_issue_count
    + documentation_issue_count
    + invalid_url_count
    + invalid_confidence_count
)

print(
    "Total schema validation issues:",
    total_validation_issues,
)

Total schema validation issues: 0


In [92]:
print("=" * 60)
print("ENRICHMENT VALIDATION SUMMARY")
print("=" * 60)

print(
    "\nRequired-field issues:",
    required_field_issue_count,
)

print(
    "Record-type rule issues:",
    record_type_issue_count,
)

print(
    "Impact-reference issues:",
    impact_reference_issue_count,
)

print(
    "Impact-pillar issues:",
    impact_pillar_issue_count,
)

print(
    "Documentation issues:",
    documentation_issue_count,
)

print(
    "Invalid source URLs:",
    invalid_url_count,
)

print(
    "Invalid confidence values:",
    invalid_confidence_count,
)

print(
    "\nTotal validation issues:",
    total_validation_issues,
)

if total_validation_issues == 0:
    print(
        "\nAll enrichment records are properly "
        "structured and documented."
    )

ENRICHMENT VALIDATION SUMMARY

Required-field issues: 0
Record-type rule issues: 0
Impact-reference issues: 0
Impact-pillar issues: 0
Documentation issues: 0
Invalid source URLs: 0
Invalid confidence values: 0

Total validation issues: 0

All enrichment records are properly structured and documented.
